In [ ]:

def processForecastTable(session,run_date,month,run_version):

    if  PRED_LEVEL=="weekly":
        test_df=session.table(TEST_FORECAST_TABLE).to_pandas()
        dates_df=session.table(RAW_TEST_TABLE).to_pandas()
        dates_df=dates_df[['Date','Regularized_Date','MONTH_DATE']].drop_duplicates()
        test_df=test_df.rename(columns={'TS':'Regularized_Date'})
        test_df=pd.merge(test_df,dates_df,on="Regularized_Date",how="left")
        test_df=test_df.groupby(['SERIES','MONTH_DATE'])['FORECAST'].sum().reset_index()
        test_df['PREDICTED_SALES']=test_df['FORECAST'].apply(lambda x: 0 if x<0 else round(x))

        test_df=test_df[['MONTH_DATE','SERIES','PREDICTED_SALES']]
        test_df.columns=['DATES','SERIES','PREDICTED_SALES']
        write_to_snowflake(session,test_df,FORECAST_TABLE,"overwrite")

        

    
    forecast_query=f'''
    SELECT * FROM {FORECAST_TABLE} WHERE DATES='{month}'
    '''
    df=session.sql(forecast_query).to_pandas()
    #df=session.table(FORECAST_TABLE).to_pandas()
    df['PARENT_DEALER_CODE']=df['SERIES'].apply(lambda x:x.split("_")[0].strip('"'))
    df['MODEL_FAMILY']=df['SERIES'].apply(lambda x:x.split("_")[1])
    df['FAMILY_CODE']=df['SERIES'].apply(lambda x:x.split("_")[2].strip('"'))

    df['UNIQUE FAMILY CODE']=df['MODEL_FAMILY']+"<>"+df['FAMILY_CODE']
    df['RUN_DATE']=run_date
    df['RUN_VERSION']=run_version
    session.create_dataframe(df).write.mode("append").save_as_table(PREDICTION_TABLE)